# 02 · Training curves and evaluation
Reads `runs/*/log.jsonl` written by `scripts/train.py` and evaluates the saved checkpoints greedily on every country.

In [ ]:
import sys, json; sys.path.insert(0, "..")
from pathlib import Path
import matplotlib.pyplot as plt
runs = sorted(p for p in Path("../runs").glob("*") if (p / "log.jsonl").exists())
logs = {p.name: [json.loads(l) for l in (p / "log.jsonl").read_text().splitlines() if l.strip()] for p in runs}
list(logs)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for name, rows in logs.items():
    it = [r["iteration"] for r in rows]
    axes[0].plot(it, [r["solve_rate"] for r in rows], label=name)
    axes[1].plot(it, [r["first_guess_acc"] for r in rows], label=name)
    axes[2].plot(it, [r["mean_guesses"] for r in rows], label=name)
for ax, t in zip(axes, ["solve rate", "first-guess accuracy", "mean guesses when solved"]):
    ax.set_title(t); ax.set_xlabel("iteration"); ax.grid(alpha=.3)
axes[1].axhline(1 / 195, color="gray", ls="--", label="chance"); axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
from flybrain_worldle.training.reinforce import evaluate, load_agent
import pandas as pd
rows = []
for p in runs:
    if (p / "best.pt").exists():
        agent, cfg, game = load_agent(p / "best.pt")
        rows.append({"run": p.name, "vision": cfg.vision, "graph": cfg.graph, **evaluate(agent, game)})
pd.DataFrame(rows).round(3)